# 02 - Exploratory Data Analysis (EDA)

## Objective

Explore variable distributions, outliers, correlations, group behavior, and temporal patterns. The results guide data preparation and the selection of a forecasting target.

In [ ]:
from pathlib import Path
import pandas as pd

# Caminho of the project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "interm"

# Load the data
csv_file = DATA_PATH / "data_1.csv"
banco = pd.read_csv(csv_file)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Parâmetros of estilo of the Gráfico based in the manual of the MBA
plt.rcParams.update({

    # font
    "font.family": "Arial",
    "font.size": 11,

    # background
    "figure.facecolor": "white",
    "axes.facecolor": "white",

    # without grid
    "axes.grid": False,

    # without title
    "axes.titlesize": 0,

    # axes
    "axes.edgecolor": "black",
    "axes.linewidth": 1.5,

    # color of the textos
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    # size of the titles of thexes
    "axes.labelsize": 11

})

In [ ]:
# dimension of the dataset
n_samples, n_features = banco.shape

print("=" * 35)
print("DATASET DIMENSIONS")
print("=" * 35)
print(f"Number of observations : {n_samples}")
print(f"Number of features   : {n_features}")

## Descriptive Statistical Analysis

In [ ]:
banco.describe()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12,5))

sns.histplot(data=banco['valor_unitario'], ax=axes[0])
sns.boxplot(data=banco['valor_unitario'], ax=axes[1])

plt.show()

In [ ]:
print(f'The variable valor_unitario has skewness of {banco['valor_unitario'].skew()}')
print(f'The variable valor_unitario has kurtosis of {banco['valor_unitario'].kurt()}')

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12,5))

sns.histplot(data=banco['valor_total'], ax=axes[0])
sns.boxplot(data=banco['valor_total'], ax=axes[1])

plt.show()

In [ ]:
print(f'The variable valor_total has skewness of {banco['valor_total'].skew()}')
print(f'The variable valor_total has kurtosis of {banco['valor_total'].kurt()}')

The unit-price and total-value distributions are positively skewed. A small number of high-value products creates long right tails, so the median and interquartile range are useful complements to the mean.

### Identifying Outliers

#### Unit Price

In [ ]:
banco.groupby('grupo')['valor_unitario'].describe()

In [ ]:
banco.loc[
    banco['valor_unitario'] > 1000,
    ['grupo','descricao','valor_unitario','quantidade','valor_total']
].sort_values('valor_unitario', ascending=False)

Unit prices vary substantially across products and groups. Large deviations from the mean are not necessarily errors because products have different price ranges and sales units.

#### Quantity

In [ ]:
banco.groupby('grupo')['quantidade'].describe()

In [ ]:
banco.loc[
    banco['quantidade'] > 50,
    ['grupo','descricao','quantidade','valor_unitario','valor_total']
].sort_values('quantidade', ascending=False)

Quantity analysis confirms that the business combines unit-based and length-based products. This explains the large variation in transaction quantities and motivates the use of multiple complementary metrics.

## Distribution by Group

In [ ]:
banco['grupo'].value_counts()

In [ ]:
banco['grupo'].value_counts(normalize=True)*100

The excluded groups represent a small share of the cleaned dataset and have irregular or poorly defined product classifications. The main analysis focuses on groups with more consistent business meaning.

### Dataset Version 2

In [ ]:
banco_int = banco.copy()
banco_int.head()

In [ ]:
banco_int.drop(['preco_custo', 'saldo'], axis=1, inplace=True)

In [ ]:
banco_int.head()

### Quantity Sold by Group

In [ ]:
quantidade_vendida = banco_int.groupby('grupo')['quantidade'].sum().sort_values(ascending=False)

plt.figure(figsize=(10,6))
plt.bar(quantidade_vendida.index, quantidade_vendida.values)
plt.title('quantity sold by group')
plt.xlabel('group')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=90)
plt.ylabel('quantity sold')
plt.show()

CABO behaves like a structural outlier because its products are often sold by length. The high quantity is legitimate and should not be treated as a data error.

### Number of Sales by Group

In [ ]:
numero_vendas = banco_int.groupby('grupo').size().sort_values(ascending=False)

plt.figure(figsize=(10,6))
plt.bar(numero_vendas.index, numero_vendas.values)
plt.title('number of sales by group')
plt.xlabel('group')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=90)
plt.ylabel('number of sales')
plt.show()

SEM GRUPO still represents a meaningful number of records. Its unclear classification makes it less useful for group-level interpretation, so it is examined separately.

### Analysis of SEM GRUPO, TV BOX, REDES, PRODUTO ESPECIAL, and RECEPTOR

In [ ]:
banco_int.loc[banco_int['grupo'] == 'SEM GRUPO']

In [ ]:
banco_int[banco_int['grupo'] == 'SEM GRUPO'].groupby('descricao').agg(
        vendas=('descricao','size'),
        quantidade=('quantidade','sum'),
        faturamento=('valor_total','sum')
    ).sort_values('revenue', ascending=False)

SEM GRUPO has high cardinality and unclear product composition. Keeping it in the main group comparison could obscure relationships between well-defined groups.

In [ ]:
banco_int.loc[banco_int['grupo'] == 'TV BOX']

In [ ]:
banco_int[banco_int['grupo'] == 'TV BOX'].groupby('descricao').agg(
        vendas=('descricao','size'),
        quantidade=('quantidade','sum'),
        faturamento=('valor_total','sum')
    ).sort_values('revenue', ascending=False)

TV BOX contains valid sales and may help explain company revenue, but it represents a specialized product segment.

In [ ]:
banco_int.loc[banco_int['grupo'] == 'PRODUTO ESPECIAL']

In [ ]:
banco_int[banco_int['grupo'] == 'PRODUTO ESPECIAL'].groupby('descricao').agg(
        vendas=('descricao','size'),
        quantidade=('quantidade','sum'),
        faturamento=('valor_total','sum')
    ).sort_values('revenue', ascending=False)

PRODUTO ESPECIAL has high cardinality and weak consistency in product type. It is similar to SEM GRUPO and can reduce the clarity of group comparisons.

In [ ]:
banco_int.loc[banco_int['grupo'] == 'REDES']

REDES contains only five records. These products may have been classified under other groups, so this category is not suitable for standalone analysis.

In [ ]:
banco_int.loc[banco_int['grupo'] == 'RECEPTOR']

In [ ]:
banco_int[banco_int['grupo'] == 'RECEPTOR'].groupby('descricao').agg(
        vendas=('descricao','size'),
        quantidade=('quantidade','sum'),
        faturamento=('valor_total','sum')
    ).sort_values('revenue', ascending=False)

### RECEPTOR Sales

### Revenue by Group

In [ ]:
faturamento_por_grupo = banco_int.groupby('grupo')['valor_total'].sum().sort_values(ascending=False)

plt.figure(figsize=(12,6))
plt.bar(faturamento_por_grupo.index, faturamento_por_grupo.values)
plt.title('revenue by group')
plt.xlabel('group')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=90)
plt.ylabel('revenue')
plt.show()

In [ ]:
# Criando the 3 gráficos in a same linha
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Gráfico 1 - quantity sold
quantidade_vendida.plot(
    kind='bar',
    ax=axes[0]
)
axes[0].set_title('quantity sold by group')
axes[0].set_xlabel('group')
axes[0].set_ylabel('quantity')
axes[0].tick_params(axis='x', rotation=90)

# Gráfico 2 - number of sales
numero_vendas.plot(
    kind='bar',
    ax=axes[1]
)
axes[1].set_title('number of sales by group')
axes[1].set_xlabel('group')
axes[1].set_ylabel('number of sales')
axes[1].tick_params(axis='x', rotation=90)

# Gráfico 3 - revenue
faturamento_por_grupo.plot(
    kind='bar',
    ax=axes[2]
)
axes[2].set_title('revenue by group')
axes[2].set_xlabel('group')
axes[2].set_ylabel('revenue (R$)')
axes[2].tick_params(axis='x', rotation=90)

# Ajusta the espaçamento
plt.tight_layout()

plt.show()

### Combined Group Analysis

The three metrics—units sold, transaction count, and revenue—reveal different group profiles. High volume does not automatically imply high revenue because sales units and price levels vary by group.

In [ ]:
banco_int.head()

In [ ]:
banco_int = banco_int[~banco_int['grupo'].isin(['SEM GRUPO', 'PRODUTO ESPECIAL', 'TV BOX', 'REDES'])]

In [ ]:
# number of records restantes in the dataset of date
print(f"Number of records remaining in the dataset:: {banco_int.shape[0]}")

### Revenue by Group and Year

In [ ]:
# Convert the date column to datetime
banco_int['data_venda'] = pd.to_datetime(
    banco_int['data_venda'],
    errors='coerce' # Convert invalid values to NaT
)

In [ ]:
faturamento_grupo_ano = banco_int.groupby([banco_int['data_venda'].dt.year,'grupo'])['valor_total'].sum().reset_index()

faturamento_grupo_ano.columns = ['ano','grupo','revenue']

pivot = faturamento_grupo_ano.pivot(
    index='ano',
    columns='grupo',
    values='revenue'
)

fig, ax = plt.subplots(figsize=(12, 6))

pivot.plot(
    ax=ax,
    marker='o',
    linewidth=2
)

# titles of thexes
ax.set_xlabel("year", fontsize=11, fontname="Arial")
ax.set_ylabel("revenue (R$)", fontsize=11, fontname="Arial")

# Remove the title
ax.set_title("")

# Remove a grid
ax.grid(False)

# Remove bordas superior and direita
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# keeps only thexes principais
ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")

# legend
ax.legend(
    title=None,
    bbox_to_anchor=(0.25, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.show()

The group analysis shows high product diversity and different sales mechanisms. This limits direct comparisons and supports analyzing seasonality at the aggregate revenue level.

### Correlation Between Sales Across Groups

In [ ]:
# Monthly revenue time series by group

faturamento_grupos = (
banco_int
    .groupby([
        banco_int['data_venda'].dt.to_period('M'), 'grupo'
    ])['valor_total'].sum().unstack(fill_value=0)
)
# unstack() cria "table dynamics" a partir of the result of the groupby, where the index is a sales records monthly 
# with groups as columns and monthly revenue as values.    
# Missing group-month combinations are filled with zero.

faturamento_grupos.index = (
    faturamento_grupos.index.to_timestamp()
)

faturamento_grupos.head()


In [ ]:
# correlation between sales monthly of the groups

corr = faturamento_grupos.corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm'
)

The strongest correlations occur among groups that are commonly used together in installations, especially ANTENA, CABO, COMPONENTES ANTENAS, and CONVERSOR. Correlation should be interpreted as co-movement, not causation.

## Average Ticket by Group

In [ ]:
resumo_grupos = banco.groupby('grupo').agg(vendas=('grupo', 'size'), quantidade=('quantidade', 'sum'), 
                                           faturamento=('valor_total', 'sum')).sort_values('revenue', ascending=False)


resumo_grupos

In [ ]:
resumo_grupos['ticket_medio'] = resumo_grupos['revenue'] / resumo_grupos['sales']

resumo_grupos.sort_values('ticket_medio', ascending=False)

ANTENA has a balanced relationship between transaction count and units sold. This suggests that most transactions include at least one antenna.

The highest average tickets are associated with RECEPTOR, INTERFONE, SEGURANCA, and PORTAO. These groups are more relevant to revenue than transaction count alone would suggest.

In [ ]:
def plot_faturamento_anual(df, grupos, titulo):
    df_filtrado = df[df["grupo"].isin(grupos)].copy()

    anual = (
        df_filtrado
        .groupby([df_filtrado["data_venda"].dt.year, "grupo"])["valor_total"]
        .sum()
        .reset_index()
        .pivot(index="data_venda", columns="grupo", values="valor_total")
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    anual.plot(ax=ax, marker="o", linewidth=2)

    ax.set_title(titulo)
    ax.set_xlabel("year")
    ax.set_ylabel("revenue (R$)")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(title=None, frameon=False)
    plt.tight_layout()
    plt.show()

#### Annual Revenue: INTERFONE, PORTAO, and SEGURANCA

In [ ]:
# intercom, PORTAO and SEGURANCA
plot_faturamento_anual(
    banco_int,
    ["INTERFONE", "PORTAO", "SEGURANCA"],
    "revenue annual: intercom, PORTAO and SEGURANCA"
)

#### Annual Revenue: ANTENA, COMPONENTES ANTENAS, and CABO

In [ ]:
plot_faturamento_anual(
    banco_int,
    ["ANTENA", "COMPONENTES ANTENAS", "CABO"],
    "revenue annual: antenna, COMPONENTES ANTENAS and cable"
)

This analysis provides additional business context. Early years emphasize antenna installations, while later revenue patterns reflect a broader product mix and changes in demand.

## Intermediate Data

In [ ]:
banco_int.to_csv(
    "../data/interm/data_2.csv",
    index=False
)

# Conclusion

The exploratory analysis confirms substantial heterogeneity across products and groups. Revenue, transaction count, quantity, and average ticket should be analyzed together. The cleaned intermediate dataset is suitable for temporal aggregation and forecasting.